# ClauseGuard: GPU training run

Trains the transformer models, evaluates them on the test split, exports them to ONNX and
packages them. The result is **one file**, `clauseguard-results-<modules>.tgz`.
Full instructions: `docs/TRAINING.md` in the repository.

**Before running (Kaggle):**
1. Settings → Accelerator → **GPU T4 x2**. Settings → **Internet on** (needs a phone-verified account).
2. Add-ons → Secrets → add `GITHUB_TOKEN` (a read-only GitHub token for the repository) and tick it for this notebook.
3. Set `MODULES` below: first run `"tos privacy"`, second run `"contract"`.
4. **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the tab.
5. When it has finished: open the version → **Output** → download the `.tgz` file.

In [ ]:
# 1. Settings
MODULES = "tos privacy"   # second run: "contract"
OWNER, REPO = "Raman-Luhach", "github.com/Raman-Luhach/clauseGuard.git"

import os, pathlib, subprocess


def secret(name):
    """A secret from Kaggle Secrets, Colab Secrets, or an environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


TOKEN = secret("GITHUB_TOKEN")
OUT = next((p for p in ("/kaggle/working", "/content") if pathlib.Path(p).exists()), os.getcwd())
print("GitHub token found:", bool(TOKEN), "| results go to", OUT, "| modules:", MODULES)

In [ ]:
# 2. Get the code. The token is used only for cloning; it is never printed or kept in git.
CODE = "/tmp/clauseguard"
if not pathlib.Path(CODE).exists():
    url = f"https://{OWNER}:{TOKEN}@{REPO}" if TOKEN else f"https://{REPO}"
    done = subprocess.run(["git", "clone", "--depth", "1", url, CODE], capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError("git clone failed: " + done.stderr.replace(TOKEN or "\0", "***"))
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Install, train, evaluate, export, package. Progress is printed as it happens.
env = {**os.environ, "MODULES": MODULES, "OUT": OUT}
process = subprocess.Popen(["bash", "scripts/train_on_gpu.sh"], cwd=CODE, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="", flush=True)
if process.wait():
    raise RuntimeError(f"failed with exit code {process.returncode}; see the log above")

In [ ]:
# 4. The file to download (Kaggle: the version's Output tab; Colab: the Files panel).
for f in sorted(pathlib.Path(OUT).glob("clauseguard-results-*.tgz")):
    print(f, f"{f.stat().st_size / 1e6:.0f} MB")